# MLflow Experiment Tracking

MLflow is an open-source platform designed to manage and track machine learning workflows.

In this project, MLflow will be used to track the final Optuna-tuned XGBoost model developed for flight price prediction.

The main objectives of using MLflow are:

- Track model hyperparameters
- Track model evaluation metrics
- Store trained model artifacts
- Maintain reproducibility of experiments
- Provide a centralized interface for reviewing model experiments

The final model selected during model development was an Optuna-tuned XGBoost Regressor.

In [4]:
import mlflow

print(mlflow.__version__)

3.15.1


In [5]:
# Importing the required libraries

import os
import json
import joblib
import mlflow
import mlflow.xgboost

## Checking Saved Model Artifacts

Before starting experiment tracking, we verify that the final model and its supporting artifacts are available.

The model and its metadata were saved during the model development stage. These artifacts will now be used to create the MLflow experiment record.

In [6]:
model_dir = "../models"

os.listdir(model_dir)

['best_hyperparameters.json',
 'feature_columns.pkl',
 'final_model_metrics.json',
 'model_metadata.json',
 'xgb_flight_price_model.pkl']

In [7]:
# Loading the model

model = joblib.load(
    "../models/xgb_flight_price_model.pkl"
)

print(type(model))

<class 'xgboost.sklearn.XGBRegressor'>


In [8]:
with open("../models/best_hyperparameters.json", "r") as f:
    best_params = json.load(f)

print(best_params)

{'n_estimators': 383, 'max_depth': 3, 'learning_rate': 0.1024932221692416, 'min_child_weight': 9, 'subsample': 0.6488152939379115, 'colsample_bytree': 0.798070764044508, 'gamma': 0.17194260557609198, 'reg_alpha': 1.527156759251193, 'reg_lambda': 0.01967432802530612}


In [9]:
with open("../models/final_model_metrics.json", "r") as f:
    final_metrics = json.load(f)

print(final_metrics)

{'MAE': 50.57661854739916, 'RMSE': 62.304069442411446, 'R2_Score': 0.978431888191669}


In [10]:
mlflow.set_experiment("Flight Price Prediction")

<Experiment: artifact_location='file:d:/DA_Preperation/Travel-Analytics-MLOps/notebooks/mlruns/1', creation_time=1787057817966, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787057817966, lifecycle_stage='active', name='Flight Price Prediction', tags={}, trace_location=None, workspace='default'>

## Starting an MLflow Run

An MLflow run represents one execution of a machine learning experiment.

The `mlflow.start_run()` function starts a new run within the selected experiment and provides a context in which parameters, metrics, tags, and artifacts can be logged.

For this project, the run will represent the final Optuna-tuned XGBoost model.

In [11]:
with mlflow.start_run(run_name="Optuna_Tuned_XGBoost"):
    print("MLFlow run started successfully")

2026/08/20 17:43:56 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/08/20 17:43:56 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

MLFlow run started successfully


## Tracking the Tuned XGBoost Model

We now create a dedicated MLflow run for the final Optuna-tuned XGBoost model.

During this run, we will record the model's hyperparameters, evaluation metrics, metadata, and trained model artifact.

This allows us to keep a complete and reproducible record of the final model.

In [14]:
mlflow.end_run()

In [15]:
mlflow.active_run()

In [16]:
with mlflow.start_run(run_name= 'Optuna_Tuned_XGBoost'):
    mlflow.log_params(best_params)
    print('MLflow run created and parameters logged successfuly')

MLflow run created and parameters logged successfuly


## Logging Model Evaluation Metrics

After recording the model hyperparameters, we log the evaluation metrics of the tuned XGBoost model.

Metrics describe the performance of the trained model on the test dataset.

For this project, we will track Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² Score.

These metrics allow us to compare model performance and identify the best-performing model.

In [17]:
import json

with open("../models/final_model_metrics.json", "r") as f:
    final_metrics = json.load(f)

final_metrics

{'MAE': 50.57661854739916,
 'RMSE': 62.304069442411446,
 'R2_Score': 0.978431888191669}

In [18]:
mlflow.log_metrics(final_metrics)

## Adding Metadata Using MLflow Tags

MLflow tags are key-value pairs used to add descriptive metadata to an experiment run.

Unlike parameters, tags do not represent model configuration. Unlike metrics, they do not represent model performance.

Tags are useful for identifying the model type, tuning technique, problem type, dataset, or other contextual information associated with a run.

In [19]:
mlflow.set_tag("model_type", "XGBoost Regressor")
mlflow.set_tag("tuning_method", "Optuna")
mlflow.set_tag("problem_type", "Regression")
mlflow.set_tag("project", "Flight Price Prediction")

In [21]:
mlflow.xgboost.log_model(model, name="xgb_flight_price_model")

## Registering the Final XGBoost Model

The final Optuna-tuned XGBoost model has already been logged to MLflow as part of an experiment run.

We now register this model in the MLflow Model Registry.

Model registration provides a centralized way to manage model versions and creates a bridge between experiment tracking and model deployment.

Instead of treating the trained model as only a file, the model can now be managed as a versioned MLflow model.

In [1]:
from mlflow import MlflowClient

client = MlflowClient()

In [2]:
registered_model_name = "flight-price-xgboost"

try:
    client.create_registered_model(registered_model_name)
    print(f"Registered model created: {registered_model_name}")
except Exception as e:
    print("Registered model may already exist:", e)

Registered model created: flight-price-xgboost


In [3]:
import mlflow
import os

print("Current working directory:")
print(os.getcwd())

print("\nMLflow tracking URI:")
print(mlflow.get_tracking_uri())

Current working directory:
d:\DA_Preperation\Travel-Analytics-MLOps\notebooks

MLflow tracking URI:
sqlite:///D:/DA_Preperation/Travel-Analytics-MLOps/notebooks/mlflow.db


In [4]:
experiment = mlflow.get_experiment_by_name("Flight Price Prediction")

print(experiment)

<Experiment: artifact_location='file:d:/DA_Preperation/Travel-Analytics-MLOps/notebooks/mlruns/1', creation_time=1787057817966, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787057817966, lifecycle_stage='active', name='Flight Price Prediction', tags={}, trace_location=None, workspace='default'>


In [6]:
runs.columns.tolist()

['run_id',
 'experiment_id',
 'status',
 'artifact_uri',
 'start_time',
 'end_time',
 'metrics.RMSE',
 'metrics.MAE',
 'metrics.R2_Score',
 'params.reg_lambda',
 'params.subsample',
 'params.min_child_weight',
 'params.learning_rate',
 'params.colsample_bytree',
 'params.n_estimators',
 'params.max_depth',
 'params.reg_alpha',
 'params.gamma',
 'tags.problem_type',
 'tags.tuning_method',
 'tags.mlflow.user',
 'tags.mlflow.source.type',
 'tags.mlflow.runName',
 'tags.project',
 'tags.model_type',
 'tags.mlflow.source.name']

In [9]:
runs = mlflow.search_runs(
    experiment_names=["Flight Price Prediction"]
)

runs[[
    "run_id",
    "tags.mlflow.runName",
    "status",
    "metrics.MAE",
    "metrics.RMSE",
    "metrics.R2_Score"
]]

,run_id,tags.mlflow.runName,status,metrics.MAE,metrics.RMSE,metrics.R2_Score
0,8c5532c1eb47486999941609bd99c5e4,sassy-calf-971,RUNNING,50.576619,62.304069,0.978432
1,01de53ed99094615905499a2f1fcbec5,Optuna_Tuned_XGBoost,FINISHED,NaN,NaN,NaN
2,e706af4222bd47eda67505368c41667c,gregarious-sow-405,FINISHED,NaN,NaN,NaN
3,013f7273760d4f948a94000641d81f2c,Optuna_Tuned_XGBoost,FINISHED,NaN,NaN,NaN
4,49f88b622cb84a06a6add30bb4932120,invincible-moose-341,RUNNING,NaN,NaN,NaN
5,1a014aae9d624af48d45927887cdd923,Optuna_Tuned_XGBoost,FINISHED,NaN,NaN,NaN


In [10]:
runs[runs["status"] == "RUNNING"]

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.RMSE,metrics.MAE,metrics.R2_Score,params.reg_lambda,...,params.reg_alpha,params.gamma,tags.problem_type,tags.tuning_method,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.runName,tags.project,tags.model_type,tags.mlflow.source.name
0,8c5532c1eb47486999941609bd99c5e4,1,RUNNING,file:d:/DA_Preperation/Travel-Analytics-MLOps/...,2026-08-20 12:21:09.948000+00:00,NaT,62.304069,50.576619,0.978432,None,...,None,None,Regression,Optuna,Tanmay Gautam,NOTEBOOK,sassy-calf-971,Flight Price Prediction,XGBoost Regressor,MLFlow.ipynb
4,49f88b622cb84a06a6add30bb4932120,1,RUNNING,file:d:/DA_Preperation/Travel-Analytics-MLOps/...,2026-08-18 13:14:36.684000+00:00,NaT,NaN,NaN,NaN,0.01967432802530612,...,1.527156759251193,0.17194260557609198,None,None,Tanmay Gautam,NOTEBOOK,invincible-moose-341,None,None,MLFlow.ipynb


In [13]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

run_id = "8c5532c1eb47486999941609bd99c5e4"

artifacts = client.list_artifacts(run_id)

for artifact in artifacts:
    print(artifact.path, artifact.is_dir)

In [15]:
import mlflow

run_id = "8c5532c1eb47486999941609bd99c5e4"

run = mlflow.get_run(run_id)

print("Run name:", run.data.tags.get("mlflow.runName"))
print("Status:", run.info.status)
print("Metrics:", run.data.metrics)
print("Parameters:", run.data.params)

Run name: sassy-calf-971
Status: RUNNING
Metrics: {'MAE': 50.57661854739916, 'RMSE': 62.304069442411446, 'R2_Score': 0.978431888191669}
Parameters: {}


In [16]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

artifacts = client.list_artifacts(run_id)

print("Number of artifacts:", len(artifacts))

if artifacts:
    for artifact in artifacts:
        print("Artifact:", artifact.path)
        print("Is directory:", artifact.is_dir)
else:
    print("No artifacts found for this run.")

Number of artifacts: 0
No artifacts found for this run.


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment = mlflow.get_experiment_by_name("Flight Price Prediction")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

print("Total runs:", len(runs))



Total runs: 6


In [18]:
for _, row in runs.iterrows():
    run_id = row["run_id"]
    
    artifacts = client.list_artifacts(run_id)
    
    if artifacts:
        print("\nRun ID:", run_id)
        print("Run Name:", row.get("tags.mlflow.runName", "N/A"))
        
        for artifact in artifacts:
            print("   Artifact:", artifact.path)

In [19]:
import joblib
import mlflow

final_model = joblib.load("../models/xgb_flight_price_model.pkl")

print(type(final_model))

<class 'xgboost.sklearn.XGBRegressor'>


In [20]:
if mlflow.active_run() is not None:
    print("Active run found:", mlflow.active_run().info.run_id)
    mlflow.end_run()

print("Active run:", mlflow.active_run())


Active run: None


In [21]:
with mlflow.start_run(run_name="Final_Tuned_XGBoost") as run:

    run_id = run.info.run_id

    print("Run ID:", run_id)
    print("Run Name:", "Final_Tuned_XGBoost")

    # Log the model
    mlflow.xgboost.log_model(
        xgb_model=final_model,
        artifact_path="model"
    )

    print("Model logged successfully!")

2026/08/22 13:16:38 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/08/22 13:16:38 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

Run ID: 3ac8ed07e2db4ec59f4e6df789f6e164
Run Name: Final_Tuned_XGBoost
Model logged successfully!


In [22]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

artifacts = client.list_artifacts(run_id)

print("Artifacts:")
for artifact in artifacts:
    print(artifact.path, artifact.is_dir)

Artifacts:


In [23]:
import mlflow
import os

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Current directory:", os.getcwd())

run = mlflow.get_run(run_id)

print("\nRun ID:", run.info.run_id)
print("Run status:", run.info.status)
print("Run name:", run.data.tags.get("mlflow.runName"))
print("Metrics:", run.data.metrics)

MLflow version: 3.15.1
Tracking URI: sqlite:///D:/DA_Preperation/Travel-Analytics-MLOps/notebooks/mlflow.db
Current directory: d:\DA_Preperation\Travel-Analytics-MLOps\notebooks

Run ID: 3ac8ed07e2db4ec59f4e6df789f6e164
Run status: FINISHED
Run name: Final_Tuned_XGBoost
Metrics: {}


In [24]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

artifacts = client.list_artifacts(run_id)

print("\nNumber of artifacts:", len(artifacts))

for artifact in artifacts:
    print(artifact.path, artifact.is_dir)


Number of artifacts: 0


In [25]:
mlflow.xgboost.log_model(
    xgb_model=final_model,
    artifact_path="model"
)

2026/08/22 13:20:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [26]:
import mlflow

run = mlflow.get_run(run_id)

print("Run ID:", run.info.run_id)
print("Artifact URI:", run.info.artifact_uri)

Run ID: 3ac8ed07e2db4ec59f4e6df789f6e164
Artifact URI: file:d:/DA_Preperation/Travel-Analytics-MLOps/notebooks/mlruns/0/3ac8ed07e2db4ec59f4e6df789f6e164/artifacts


In [27]:
import os

artifact_uri = run.info.artifact_uri

print("Artifact URI:")
print(artifact_uri)

Artifact URI:
file:d:/DA_Preperation/Travel-Analytics-MLOps/notebooks/mlruns/0/3ac8ed07e2db4ec59f4e6df789f6e164/artifacts


In [28]:
mlflow.xgboost.log_model(
    xgb_model=final_model,
    name="xgb_flight_price_model"
)

In [29]:
if mlflow.active_run() is not None:
    mlflow.end_run()

print("Active run:", mlflow.active_run())

Active run: None


In [30]:
import mlflow

with mlflow.start_run(run_name="Final_Tuned_XGBoost_v2") as run:

    # Log final model parameters
    mlflow.log_params({
        "n_estimators": 383,
        "max_depth": 3,
        "learning_rate": 0.1024932221692416,
        "min_child_weight": 9,
        "subsample": 0.6488152939379115,
        "colsample_bytree": 0.798070764044508,
        "gamma": 0.171942605576092,
        "reg_alpha": 1.527156759251193,
        "reg_lambda": 0.01967432802530612
    })

    # Log final evaluation metrics
    mlflow.log_metrics({
        "MAE": 50.57661854739916,
        "RMSE": 62.304069442411446,
        "R2_Score": 0.978431888191669
    })

    # Log the XGBoost model using MLflow 3.x syntax
    model_info = mlflow.xgboost.log_model(
        xgb_model=final_model,
        name="xgb_flight_price_model"
    )

    print("Run ID:", run.info.run_id)
    print("Model ID:", model_info.model_id)
    print("Model URI:", model_info.model_uri)

Run ID: baca12a95b3649738869684cb28ae201
Model ID: m-d16f013d8c7b43629495385c120c60a7
Model URI: models:/m-d16f013d8c7b43629495385c120c60a7


In [31]:
client.list_artifacts(run_id)

[]

In [32]:
logged_model = mlflow.get_logged_model(model_info.model_id)

print("Model ID:", logged_model.model_id)
print("Model Name:", logged_model.name)
print("Model Type:", logged_model.model_type)
print("Status:", logged_model.status)
print("Source Run ID:", logged_model.source_run_id)

Model ID: m-d16f013d8c7b43629495385c120c60a7
Model Name: xgb_flight_price_model
Model Type: None
Status: READY
Source Run ID: baca12a95b3649738869684cb28ae201


In [33]:
registered_model = mlflow.register_model(
    model_uri=f"models:/{model_info.model_id}",
    name="flight-price-xgboost"
)

print("Registered Model:", registered_model.name)
print("Model Version:", registered_model.version)

Registered Model: flight-price-xgboost
Model Version: 1


Registered model 'flight-price-xgboost' already exists. Creating a new version of this model...
Created version '1' of model 'flight-price-xgboost'.


## Verifying the Registered Model

The final tuned XGBoost model has been registered in the MLflow Model Registry as `flight-price-xgboost`, Version 1.

The registered version provides a stable reference to the model that can later be loaded for deployment and managed independently from the original experimentation run.

In [34]:
from mlflow import MlflowClient

client = MlflowClient()

versions = client.search_model_versions(
    "name='flight-price-xgboost'"
)

for version in versions:
    print("Model Name:", version.name)
    print("Version:", version.version)
    print("Run ID:", version.run_id)
    print("Status:", version.status)
    print("-" * 50)

Model Name: flight-price-xgboost
Version: 1
Run ID: baca12a95b3649738869684cb28ae201
Status: READY
--------------------------------------------------
